<a href="https://colab.research.google.com/github/CaoTrongNghia/dipoleMoment-dftEstimator/blob/non-drive-mounted-dft/dft_spice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torch_geometric

# Clone TorchMD-NET fresh
!git clone https://github.com/torchmd/torchmd-net.git
%cd torchmd-net

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 43.7 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.6 MB/s eta 0:00:00
Cloning into 'torchmd-net'...
remote: Enumerating objects: 8780, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 8780 (delta 398), reused 371 (delta 371), pack-reused 8351 (from 4)
Receiving objects: 100% (8780/8780), 189.01 MiB | 23.51 MiB/s, done.
Resolving deltas: 100% (6125/6125), done.
/content/torchmd-net


In [ ]:
# Install TorchMD-NET
!pip install -e . --no-cache-dir --default-timeout=200

# torchvision and torchaudio are not needed

Obtaining file:///content/torchmd-net
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 11.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.7/138.7 MB 259.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 245.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 324.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 266.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 242.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.

In [ ]:
!pip install h5py ase
!pip install wandb

In [ ]:
import sys
sys.path.append("/usr/local/lib/python3.11/site-packages/")
sys.path.append("/content/torchmd-net")

In [ ]:
import urllib.request
import os

# Download SPICE 1.1.4 from Zenodo
url = "https://zenodo.org/records/8222043/files/SPICE-1.1.4.hdf5"
filename = "SPICE.hdf5"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)
    print("Download complete!")
else:
    print("SPICE dataset already exists.")

Download complete!


In [ ]:
import h5py

with h5py.File('/content/SPICE.hdf5', 'r') as f:
    # Grab the very first molecule in the dataset
    first_molecule = list(f.keys())[0]
    print(f"Keys inside {first_molecule}:")
    print(list(f[first_molecule].keys()))

Keys inside 103147721:
['atomic_numbers', 'conformations', 'dft_total_energy', 'dft_total_gradient', 'formation_energy', 'mayer_indices', 'mbis_charges', 'mbis_dipoles', 'mbis_octupoles', 'mbis_quadrupoles', 'scf_dipole', 'scf_quadrupole', 'smiles', 'subset', 'wiberg_lowdin_indices']


In [ ]:
import h5py
import torch
from torch_geometric.data import Data
import numpy as np

def process_spice_hdf5(file_path):
    processed_graphs = []

    # 1. Open the raw SPICE HDF5 dataset safely
    with h5py.File(file_path, 'r') as f:
        # Loop through each distinct molecule group in the dataset
        for mol_name in f.keys():
            mol_group = f[mol_name]

            # Extract atomic numbers (Z-matrix).
            # Cast to torch.long because these are categorical indices for embeddings
            atomic_numbers = torch.tensor(mol_group['atomic_numbers'][:], dtype=torch.long)

            # Pull arrays containing all geometric variations (conformations)
            # Shapes are typically: (num_conformations, num_atoms, 3) for geometry/forces
            all_coords = mol_group['conformations'][:]
            all_forces = mol_group['dft_total_gradient'][:]
            all_energies = mol_group['formation_energy'][:] # or 'total_energy' depending on target

            num_conformations = all_coords.shape[0]

            # 2. Inner loop: Extract every individual slice as a unique graph
            for i in range(num_conformations):
                # Isolate the i-th conformation properties
                pos = torch.tensor(all_coords[i], dtype=torch.float32)
                neg_dy = torch.tensor(all_forces[i], dtype=torch.float32)
                y = torch.tensor([all_energies[i]], dtype=torch.float32)

                # Check for skipped or corrupted steps in raw DFT data
                if torch.isnan(pos).any() or torch.isnan(y).any():
                    continue

                # 3. Create the PyTorch Geometric Graph Object
                # TorchMD-Net explicitly looks for 'z', 'pos', 'y', and 'neg_dy'
                molecule_graph = Data(
                    z=atomic_numbers,
                    pos=pos,
                    y=y,
                    neg_dy=neg_dy
                )

                processed_graphs.append(molecule_graph)

    return processed_graphs

# Execution block
raw_hdf5_path = "/content/SPICE.hdf5"
dataset = process_spice_hdf5(raw_hdf5_path)

# Save it to disk so you never have to parse the heavy HDF5 file again
torch.save(dataset, "spice_processed_graphs.pt")
print(f"Successfully processed {len(dataset)} molecular configurations!")

Successfully processed 1110165 molecular configurations!


In [ ]:
from torch_geometric.loader import DataLoader
import torch

# Load the graphs back into memory
dataset = torch.load("/content/spice_processed_graphs.pt", weights_only=False)

# Perform a quick 80/20 train/validation split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Create specialized PyG DataLoaders to stack variable-sized molecules into single batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torchmdnet.models.model import create_model

# 1. THE SECRET WEAPON: A Custom Dictionary that absorbs KeyErrors!
class BulletproofConfig(dict):
    def __getitem__(self, key):
        # If TorchMD-Net's source code asks for a key we didn't provide,
        # intercept the crash and return a safe, neutral default!
        if key not in self:
            if key == 'atom_filter': return -1
            if key == 'layernorm_on_vec': return 'whitened'
            if key == 'aggr': return 'add'
            return None
        return super().__getitem__(key)

# 2. Define the exact network parameters we want
model_config = BulletproofConfig({
    'model': 'equivariant-transformer',
    'embedding_dimension': 128,
    'num_layers': 6,
    'num_rbf': 32,
    'rbf_type': 'gauss',
    'trainable_rbf': False,
    'activation': 'silu',
    'attn_activation': 'silu',
    'num_heads': 8,
    'distance_influence': 'both',
    'cutoff_lower': 0.0,
    'cutoff_upper': 5.0,
    'max_num_neighbors': 32,
    'neighbor_embedding': True,
    'derivative': True,          # Crucial for calculating forces
    'max_z': 100,
    'output_model': 'Scalar',
    'precision': 32,
    'reduce_op': 'add',
    'prior_model': None

    'atom_filter': -1,                  # Don't filter out elements
    'layernorm_on_vec': 'whitened',     # Keeps the directional vectors stable
    'aggr': 'add'                       # Aggregates graph nodes properly
})

# 3. Compile the model safely on your GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_model(model_config).to(device)

print(f"Success! Model compiled on {device} without a single KeyError.")

# 4. Initialize Optimizer
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-8)

# 5. Main Training Function
def train_one_epoch():
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Explicitly instruct PyTorch to track position variables for force calculations
        batch.pos.requires_grad_(True)

        # Run forward pass through TorchMD-Net
        pred_energy, pred_forces = model(batch.z, batch.pos, batch.batch)

        # Compute Multi-Objective Loss (Energy Loss + Force Loss)
        loss_energy = F.mse_loss(pred_energy.squeeze(), batch.y.squeeze())
        loss_forces = F.mse_loss(pred_forces, batch.neg_dy)

        # Weigh force information higher
        combined_loss = loss_energy + (10.0 * loss_forces)

        # Backpropagation step
        combined_loss.backward()
        optimizer.step()

        total_loss += combined_loss.item() * batch.num_graphs

    return total_loss / len(train_loader.dataset)

# 6. Simple Validation Check Loop
def validate():
    model.eval()
    total_val_loss = 0.0

    for batch in val_loader:
        batch = batch.to(device)
        batch.pos.requires_grad_(True)

        pred_energy, pred_forces = model(batch.z, batch.pos, batch.batch)

        loss_energy = F.mse_loss(pred_energy.squeeze(), batch.y.squeeze())
        loss_forces = F.mse_loss(pred_forces, batch.neg_dy)
        combined_loss = loss_energy + (10.0 * loss_forces)

        total_val_loss += combined_loss.item() * batch.num_graphs

    return total_val_loss / len(val_loader.dataset)

# 7. Run the Training Suite
num_epochs = 50
for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch()
    val_loss = validate()
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Success! Model compiled on cuda without a single KeyError.
Module neighbors_brute_specialized_fwd_P0_T1_L1_float32 dbbff42 load on device 'cuda:0' took 865.83 ms  (compiled)


/content/torchmd-net/torchmdnet/models/utils.py:638: UserWarning: Skipping gradients for 1 atoms due to vector features being zero. This is likely due to atoms being outside the cutoff radius of any other atom. These atoms will not interact with any other atom unless you change the cutoff.
  warnings.warn(
/content/torchmd-net/torchmdnet/models/utils.py:638: UserWarning: Skipping gradients for 2 atoms due to vector features being zero. This is likely due to atoms being outside the cutoff radius of any other atom. These atoms will not interact with any other atom unless you change the cutoff.
  warnings.warn(
/content/torchmd-net/torchmdnet/models/utils.py:638: UserWarning: Skipping gradients for 3 atoms due to vector features being zero. This is likely due to atoms being outside the cutoff radius of any other atom. These atoms will not interact with any other atom unless you change the cutoff.
  warnings.warn(
/content/torchmd-net/torchmdnet/models/utils.py:638: UserWarning: Skipping g

Epoch 01 | Train Loss: 0.0186 | Val Loss: 0.0041
